In [5]:
from langgraph.constants import START,END
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState
from langchain_deepseek import ChatDeepSeek
from langchain.messages import HumanMessage
from dotenv import load_dotenv

# 加载.env文件中的配置
load_dotenv(override=True)

#  定义模型
model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 定义状态
class OverAllState(MessagesState):
    username:str
    output:str

# 定义节点
def node_a(state:OverAllState)->OverAllState:
    return{
        "messages":[HumanMessage("你好，我是"+state["username"])],
    }

# 定义大模型节点
def llm_node(state:OverAllState)->OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages":[res],
        "output":res.content,
    }

# 构件图
builder = StateGraph(state_schema=OverAllState)

# 构建节点
builder.add_node("node_a", node_a)
builder.add_node("llm_node", llm_node)
# 构建边
builder.add_edge(START,"node_a")
builder.add_edge("node_a", "llm_node")
builder.add_edge("llm_node", END)

# 获取图

graph = builder.compile()

# 运行图
result = graph.invoke({"username":"晴曛"})
print(result)


{'messages': [HumanMessage(content='你好，我是晴曛', additional_kwargs={}, response_metadata={}, id='ebb14eb4-a67e-4d91-acc2-4205fc385882'), AIMessage(content='你好，晴曛！很高兴认识你！☀️\n\n这个名字真好听，“晴曛”两个字都带着阳光的暖意——“晴”是晴朗明媚，“曛”是日落时的余晖，组合在一起有种温暖又诗意的感觉。\n\n有什么我可以帮你的吗？无论是聊天、解答问题，还是想一起探讨什么有趣的话题，我都很乐意陪你聊聊！😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 10, 'total_tokens': 92, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '85f3e486-26af-4560-bba5-aaaa29f64295', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a135a8-5ae5-7672-bb91-a3f7fba26a6a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 82, 'total_tokens': 92, 'input_token_details': {'cach